In [0]:
%run /Workspace/Allianz/Allianz_COE/ETL_Ingestion/utils/utils_nb

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
dbutils.widgets.text("target_table_name", "")
dbutils.widgets.text("catalog_name", "allianz_coe")
dbutils.widgets.text("gold_schema_name", "gold_c360")
dbutils.widgets.text("batch_run_id", "")
dbutils.widgets.text("task_run_id", "")
dbutils.widgets.text("control_schema_name", "audit_control")
dbutils.widgets.text("vault_schema_name", "dvault_faker01")

dbutils.widgets.text("watermark_table_name", "application_watermark")

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
ip_fact_name = dbutils.widgets.get("target_table_name")
catalog_name = dbutils.widgets.get("catalog_name")
gold_schema_name = dbutils.widgets.get("gold_schema_name")
batch_run_id = dbutils.widgets.get("batch_run_id")
task_run_id = dbutils.widgets.get("task_run_id")
control_schema_name = dbutils.widgets.get("control_schema_name")
watermark_table_name = dbutils.widgets.get("watermark_table_name")
vault_schema_name = dbutils.widgets.get("vault_schema_name")

print("fact name: ", ip_fact_name)
full_table_name = f"{catalog_name}.{gold_schema_name}.{ip_fact_name}"

etl_component_name = dbutils.notebook.entry_point.getDbutils() \
.notebook().getContext().notebookPath().get()
print(etl_component_name)
print(vault_schema_name)

fact name:  fact_policy
/Allianz/Allianz_COE/ETL_Ingestion/notebooks/fact/fact_loader


INFO:py4j.clientserver:Received command c on object id p0


In [0]:
%run /Workspace/Allianz/Allianz_COE/ETL_Ingestion/config/fact_config_llm

INFO:py4j.clientserver:Received command c on object id p0


INFO:py4j.clientserver:Received command c on object id p0


INFO:py4j.clientserver:Received command c on object id p0


INFO:py4j.clientserver:Received command c on object id p0


INFO:py4j.clientserver:Received command c on object id p0


In [0]:
ALL_FACT_CONFIGS = {
    "fact_lead": FACT_LEAD_CFG,
    "fact_policy": FACT_POLICY_CFG,
    "fact_quote": FACT_QUOTE_CFG
}

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logging.info("Pipeline started")

def load_facts(fact_key, cfg):
    target_table = cfg["target_table"]
    stage_sql = cfg["stage_sql"].strip().rstrip(";")
    inc_keys = cfg.get("increment_keys")

    logging.info(f"FACT: {fact_key}")
    logging.info(f"TARGET: {target_table}")

    get_wm_query = f"""SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
    FROM {catalog_name}.{control_schema_name}.{watermark_table_name} WHERE table_name = '{full_table_name}'"""

    # Execute query
    wm_df = spark.sql(get_wm_query)

    # Get the watermark value as Python datetime
    watermark_value = wm_df.collect()[0]['watermark']

    log_message = f"stage_sql for {full_table_name} : {stage_sql}"
    task_log(task_run_id=task_run_id,task_name=full_table_name,etl_component_name=etl_component_name,message=log_message)

    if not inc_keys:
        # Full/append load
        load_sql = f"""
            INSERT INTO {target_table}
            {stage_sql}
        """
        # logging.info('----- load_sql -------')
        # logging.info(load_sql)
        spark.sql(load_sql)
        print(f"✅ Loaded {fact_key} into {target_table}")
        return

    # Incremental load via MERGE
    key_col = inc_keys[0]

    # ---------------------------------------
    # Get columns from target table
    # ---------------------------------------
    target_cols = spark.table(target_table).columns
    exclude_cols = {"load_ts", "created_ts", "created_by"}
    hash_cols = [c for c in target_cols if c not in exclude_cols]

    # ---------------------------------------
    # Build hash expressions
    # ---------------------------------------
    src_hash_expr = (
        "sha2(concat_ws('||',"
        + ",".join([f"coalesce(cast(s.`{c}` as string),'')" for c in hash_cols])
        + "),256)"
    )
    tgt_hash_expr = (
        "sha2(concat_ws('||',"
        + ",".join([f"coalesce(cast(t.`{c}` as string),'')" for c in hash_cols])
        + "),256)"
    )

    # ---------------------------------------
    # Stage dataset with hash
    # ---------------------------------------
    stage_with_hash = f"""
        SELECT
            s.*,
            {src_hash_expr} AS row_hash
        FROM (
            {stage_sql}
        ) s
    """

    log_message = f"stage_with_hash for {full_table_name} : {stage_with_hash}"
    task_log(task_run_id=task_run_id,task_name=full_table_name,etl_component_name=etl_component_name,message=log_message)

    # ---------------------------------------
    # MERGE logic
    # Insert when:
    #   - key not found
    #   - key exists but data changed
    # ---------------------------------------
    merge_sql = f"""
        MERGE INTO {target_table} t
        USING (
            {stage_with_hash}
        ) s
        ON t.{key_col} = s.{key_col}
        AND {tgt_hash_expr} = s.row_hash
        WHEN NOT MATCHED THEN INSERT *
    """
    log_message = f"merge_sql for {full_table_name} : {merge_sql}"
    task_log(task_run_id=task_run_id,task_name=full_table_name,etl_component_name=etl_component_name,message=log_message)

    # logging.info('----- merge_sql -------')
    # logging.info(merge_sql)

    spark.sql(merge_sql)
    print(f"✅ Hash-based incremental load completed for {target_table}")

    history = spark.sql(f"DESCRIBE HISTORY {target_table}").first()
    metrics = history.operationMetrics
    row_count = int(metrics['numTargetRowsInserted']) + int(metrics['numTargetRowsUpdated'])
    insert_dict = {"row_count": row_count, "watermark_used": str(watermark_value)}
    print(f"✅ Merged {fact_key} into {target_table}")

    return insert_dict

INFO:py4j.clientserver:Received command c on object id p0
INFO:root:Pipeline started


In [0]:
import json

dd = dict(ALL_FACT_CONFIGS)

try:
    if ip_fact_name in ALL_FACT_CONFIGS :
        for k, v in dd.items():
            if k == ip_fact_name:
                result_inserts = load_facts(k, v)
                task_status = {"task_current_status" : "success"}
        outputs = result_inserts | task_status
    else:
        print(f"=== Configuration not available for {ip_fact_name} ===")
        print("Available facts:", list(ALL_FACT_CONFIGS.keys()))
    
        # Construct a dictionary with multiple outputs
        outputs = {
            "task_current_status": 'failed',
            "row_count": -1,
            "notebook_name":notebook_name,
        }

        raise Exception("Entry not found in configuration !!")

except Exception as e:
    task_status = {"task_current_status" : "failed"}
    # exception_message = e
    error_details = {"exception_message": str(e)}
    notebook_details = {"notebook_name":notebook_name}
    insert_dict = {"row_count": -1}
    outputs = task_status | error_details | notebook_details | insert_dict

print(json.dumps(outputs))
# Return outputs as a JSON string
dbutils.notebook.exit(json.dumps(outputs))